In [1]:
import altair as alt
import gcsfs
import pandas as pd


GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [2]:
GCS = "gs://calitp-analytics-data/data-analyses/ntd/"
orig_df = pd.read_parquet(
    f"{GCS}raw_transit_performance_metrics_data.parquet",
    filesystem = gcsfs.GCSFileSystem()
)
orig_df.dtypes

agency_name           object
agency_status         object
city                  object
mode                  object
service               object
ntd_id                object
reporter_type         object
reporting_module      object
state                 object
primary_uza_name      object
year                  object
upt                    int64
vrh                    int64
vrm                    int64
opexp_total            int64
RTPA                  object
_merge              category
dtype: object

In [3]:
orig_df.head(2)

,agency_name,agency_status,city,mode,service,ntd_id,reporter_type,reporting_module,state,primary_uza_name,year,upt,vrh,vrm,opexp_total,RTPA,_merge
0,City of Porterville (COLT) - Transit Department,Active,Porterville,Demand Response,Purchased Transportation,90198,Building Reporter,Urban,CA,"Porterville, CA",2019,13112,2997,43696,572799,Tulare County Association of Governments,both
1,City of Porterville (COLT) - Transit Department,Active,Porterville,Demand Response,Purchased Transportation,90198,Building Reporter,Urban,CA,"Porterville, CA",2020,11523,3669,48138,686165,Tulare County Association of Governments,both


# what columns are needed
* RTPA - shows `rtpa_name`, not `rtpa_name_split`
* just read in the subset of columns needed, these metrics are precalculated

In [4]:
crosswalk = pd.read_parquet(
    f"{GCS_FILE_PATH}crosswalk2.parquet", 
    filesystem=gcsfs.GCSFileSystem(),
    columns = ["ntd_id_2022", "rtpa_name"]
).rename(columns = {"ntd_id_2022": "ntd_id"})

df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual.parquet",
    # should only certain columns be read in? now this table is much larger
    filesystem=gcsfs.GCSFileSystem(),
    columns = [
        "source_agency", "agency_status", "source_city", 
        "mode", "type_of_service", "ntd_id", 
        "reporter_type", "reporting_module", "source_state", "primary_uza_name",
        "year", "unlinked_passenger_trips", "vehicle_revenue_hours", "vehicle_revenue_miles",
        "operating_expenses_total",
    ]
).merge(
    crosswalk,
    on = "ntd_id",
    how = "left"
)

# mode should be mode_full_name
# service refers to type_of_service_full name 
df.dtypes

source_agency               object
agency_status               object
source_city                 object
mode                        object
type_of_service             object
ntd_id                      object
reporter_type               object
reporting_module            object
source_state                object
primary_uza_name            object
year                         Int64
unlinked_passenger_trips     Int64
vehicle_revenue_hours        Int64
vehicle_revenue_miles        Int64
operating_expenses_total     Int64
rtpa_name                   object
dtype: object

In [5]:
df2 = df[df.year <= 2023].reset_index(drop=True)

In [6]:
orig_df.shape, df2.shape

((2091, 17), (2676, 16))

In [ ]:
# there are some additional ones in the dbt model
m1 = pd.merge(
    orig_df[["ntd_id"]].drop_duplicates(),
    df2[["ntd_id"]].drop_duplicates(),
    on = ["ntd_id", ],
    how = "outer",
    indicator=True
)
    
m1._merge.value_counts()

In [ ]:
m1[m1._merge == "right_only"]

## These still get aggregated by agency/mode/tos and plotted

In [7]:
import B3_ntd_utils as ntd_utils

val_cols = [
    "opex_per_vrh",
    "opex_per_vrm",
    "upt_per_vrh",
    "upt_per_vrm",
    "opex_per_upt",
]

label_dict ={
    'unlinked_passenger_trips':"Unlinked Passenger Trips",
    'vehicle_revenue_miles':"Vehicle Revenue Miles",
    'vehicle_revenue_hours':"Vehicle Revenue Hours",
    'operating_expenses_total':"Operating Expense Total",
    'opex_per_vrh':"Operating Expense per Vehicle Revenue Hours",
    'opex_per_vrm':"Operating Expense per Vehicle Revenue Miles",
    'opex_per_upt':"Operating Expense per Unlinked Passenger Trips",
    'upt_per_vrh':"Unlinked Passenger Trips per Vehicle Revenue Hours",
    'upt_per_vrm':"Unlinked Passenger Trips per Vehicle Revenue Miles",
}

def make_long(df: pd.DataFrame, group_cols: list, value_cols: list):
    """
    melts dataframes to get all the metrics into a single column for better charting
    do the labeling here too
    this function can sit in notebook
    """
    df_long = df[group_cols + value_cols].melt(
        id_vars=group_cols,
        value_vars=value_cols,
    )

    df_long = df_long.assign(
        label = df_long.variable.map(label_dict)
    )

    return df_long

by_agency = ntd_utils.calculate_efficiency_metrics_by_group(
    df[df.rtpa_name=="Metropolitan Transportation Commission"], 
    ["ntd_id", "source_agency", "rtpa_name", "year"]
)
    
by_agency_long = by_agency.pipe(
    make_long, 
    ["ntd_id", "source_agency", "rtpa_name", "year"], 
    val_cols
)
# agency_name...is this renamed from source_agency?

In [8]:
cost_efficiency = ["opex_per_vrh", "opex_per_vrm", "opex_per_upt"]
service_effectiveness = ["upt_per_vrh", "upt_per_vrm"]

In [9]:
from great_tables import GT
import gt_extras as gte
import polars as pl

In [10]:
def make_wide_for_nanoplot(df, group_cols = ["ntd_id", "source_agency", "rtpa_name"], value_cols = []):
    df2 = (
        df
        .sort_values(group_cols + ["year"])
        .groupby(group_cols)
        .agg({
            
            c: lambda x: list(x) for c in ["year"] + value_cols
        })
        .reset_index()
    )
    
    df_pl = pl.from_pandas(df2)

    return df_pl
    

In [11]:
by_agency[by_agency.ntd_id=="90094"]

,ntd_id,source_agency,rtpa_name,year,unlinked_passenger_trips,vehicle_revenue_miles,vehicle_revenue_hours,operating_expenses_total,opex_per_vrh,opex_per_vrm,upt_per_vrh,upt_per_vrm,opex_per_upt
77,90094,Metropolitan Transportation Commission (MTC) -...,Metropolitan Transportation Commission,2018,0,0,0,0,NaN,NaN,NaN,NaN,NaN
78,90094,Metropolitan Transportation Commission (MTC) -...,Metropolitan Transportation Commission,2019,251570,1370339,39634,1015828,25.63,0.74,6.35,0.18,4.04
79,90094,Metropolitan Transportation Commission (MTC) -...,Metropolitan Transportation Commission,2020,419614,2434149,67326,1836307,27.27,0.75,6.23,0.17,4.38
80,90094,Metropolitan Transportation Commission (MTC) -...,Metropolitan Transportation Commission,2021,365827,4950736,104645,2361793,22.57,0.48,3.5,0.07,6.46
81,90094,Metropolitan Transportation Commission (MTC) -...,Metropolitan Transportation Commission,2022,848736,10312776,213803,5491767,25.69,0.53,3.97,0.08,6.47
82,90094,Metropolitan Transportation Commission (MTC) -...,Metropolitan Transportation Commission,2023,1113315,13422183,283315,7634546,26.95,0.57,3.93,0.08,6.86
83,90094,Metropolitan Transportation Commission (MTC) -...,Metropolitan Transportation Commission,2024,1220897,14665797,307094,8648341,28.16,0.59,3.98,0.08,7.08


In [12]:
by_agency_wide = make_wide_for_nanoplot(
    by_agency, group_cols = ["ntd_id", "source_agency", "rtpa_name"], 
    value_cols = val_cols
)

In [19]:
(
    GT(by_agency_wide)
    .cols_hide(["year", "rtpa_name"])
    .fmt_nanoplot(
        columns="opex_per_vrh", plot_type="line", missing_vals="gap" 
    ).fmt_nanoplot(
        columns="opex_per_vrm", plot_type="line", missing_vals="gap" 
    ).fmt_nanoplot(
        columns="opex_per_upt", plot_type="line", missing_vals="gap" 
    ).fmt_nanoplot(
        columns="upt_per_vrh", plot_type="line", missing_vals="gap" 
    ).fmt_nanoplot(
        columns="upt_per_vrm", plot_type="line", missing_vals="gap" 
    )
)

GT(_tbl_data=shape: (23, 9)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ ntd_id ┆ source_age ┆ rtpa_name  ┆ year      ┆ … ┆ opex_per_ ┆ upt_per_v ┆ upt_per_v ┆ opex_per_ │
│ ---    ┆ ncy        ┆ ---        ┆ ---       ┆   ┆ vrm       ┆ rh        ┆ rm        ┆ upt       │
│ str    ┆ ---        ┆ str        ┆ list[i64] ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│        ┆ str        ┆            ┆           ┆   ┆ list[f64] ┆ list[f64] ┆ list[f64] ┆ list[f64] │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 90003  ┆ San        ┆ Metropolit ┆ [2018,    ┆ … ┆ [8.4,     ┆ [58.35,   ┆ [1.66,    ┆ [5.06,    │
│        ┆ Francisco  ┆ an Transpo ┆ 2019, …   ┆   ┆ 8.45, …   ┆ 56.07, …  ┆ 1.61, …   ┆ 5.25, …   │
│        ┆ Bay Area   ┆ rtation    ┆ 2024]     ┆   ┆ 12.06]    ┆ 23.53]    ┆ 0.75]     ┆ 16.08]    │
│        ┆ Rapid T…   ┆ Co…        ┆           ┆   ┆           ┆           ┆           ┆           │
│ 90009  ┆ San Mateo  ┆ Metropolit ┆ [2018,    ┆ … ┆ [14.18,   ┆ [14.05,   ┆ [1.21,    ┆ [11.69,   │
│        ┆ County     ┆ an Transpo ┆ 2019, …   ┆   ┆ 15.8, …   ┆ 13.4, …   ┆ 1.16, …   ┆ 13.64, …  │
│        ┆ Transit    ┆ rtation    ┆ 2024]     ┆   ┆ 25.84]    ┆ 13.87]    ┆ 1.23]     ┆ 21.02]    │
│        ┆ Distr…     ┆ Co…        ┆           ┆   ┆           ┆           ┆           ┆           │
│ 90013  ┆ Santa      ┆ Metropolit ┆ [2018,    ┆ … ┆ [16.45,   ┆ [19.49,   ┆ [1.5,     ┆ [10.94,   │
│        ┆ Clara      ┆ an Transpo ┆ 2019, …   ┆   ┆ 16.66, …  ┆ 18.76, …  ┆ 1.45, …   ┆ 11.49, …  │
│        ┆ Valley Tra ┆ rtation    ┆ 2024]     ┆   ┆ 21.97]    ┆ 15.52]    ┆ 1.25]     ┆ 17.53]    │
│        ┆ nsportat…  ┆ Co…        ┆           ┆   ┆           ┆           ┆           ┆           │
│ 90014  ┆ Alameda-Co ┆ Metropolit ┆ [2018,    ┆ … ┆ [16.41,   ┆ [21.46,   ┆ [1.95,    ┆ [8.41,    │
│        ┆ ntra Costa ┆ an Transpo ┆ 2019, …   ┆   ┆ 17.32, …  ┆ 21.75, …  ┆ 1.97, …   ┆ 8.79, …   │
│        ┆ Transit D… ┆ rtation    ┆ 2024]     ┆   ┆ 24.52]    ┆ 18.62]    ┆ 1.74]     ┆ 14.13]    │
│        ┆            ┆ Co…        ┆           ┆   ┆           ┆           ┆           ┆           │
│ 90015  ┆ City and   ┆ Metropolit ┆ [2018,    ┆ … ┆ [30.91,   ┆ [59.52,   ┆ [8.1,     ┆ [3.82,    │
│        ┆ County of  ┆ an Transpo ┆ 2019, …   ┆   ┆ 32.28, …  ┆ 62.82, …  ┆ 8.42, …   ┆ 3.83, …   │
│        ┆ San        ┆ rtation    ┆ 2024]     ┆   ┆ 44.83]    ┆ 46.59]    ┆ 6.38]     ┆ 7.03]     │
│        ┆ Francis…   ┆ Co…        ┆           ┆   ┆           ┆           ┆           ┆           │
│ …      ┆ …          ┆ …          ┆ …         ┆ … ┆ …         ┆ …         ┆ …         ┆ …         │
│ 90213  ┆ City of    ┆ Metropolit ┆ [2018,    ┆ … ┆ [8.93,    ┆ [12.5,    ┆ [1.13,    ┆ [7.9,     │
│        ┆ Petaluma - ┆ an Transpo ┆ 2019, …   ┆   ┆ 9.42, …   ┆ 12.66, …  ┆ 1.15, …   ┆ 8.21, …   │
│        ┆ Transit    ┆ rtation    ┆ 2024]     ┆   ┆ 16.48]    ┆ 11.05]    ┆ 1.07]     ┆ 15.39]    │
│        ┆            ┆ Co…        ┆           ┆   ┆           ┆           ┆           ┆           │
│ 90225  ┆ San        ┆ Metropolit ┆ [2018,    ┆ … ┆ [80.22,   ┆ [139.54,  ┆ [6.66,    ┆ [12.05,   │
│        ┆ Francisco  ┆ an Transpo ┆ 2019, …   ┆   ┆ 96.63, …  ┆ 148.03, … ┆ 7.52, …   ┆ 12.85, …  │
│        ┆ Bay Area   ┆ rtation    ┆ 2024]     ┆   ┆ 113.98]   ┆ 85.89]    ┆ 4.51]     ┆ 25.26]    │
│        ┆ Water E…   ┆ Co…        ┆           ┆   ┆           ┆           ┆           ┆           │
│ 90232  ┆ Solano     ┆ Metropolit ┆ [2018,    ┆ … ┆ [7.96,    ┆ [12.08,   ┆ [0.8,     ┆ [9.95,    │
│        ┆ County     ┆ an Transpo ┆ 2019, …   ┆   ┆ 8.6, …    ┆ 12.69, …  ┆ 0.88, …   ┆ 9.75, …   │
│        ┆ Transit    ┆ rtation    ┆ 2024]     ┆   ┆ 10.1]     ┆ 9.96]     ┆ 0.52]     ┆ 19.58]    │
│        ┆            ┆ Co…        ┆           ┆   ┆           ┆           ┆           ┆           │
│ 90234  ┆ Marin      ┆ Metropoli

In [ ]:
(alt.Chart(by_agency_long)
 .mark_line()
 .encode(
     x="year",
     y="value:Q",
     color="source_agency"
 ).facet("variable", columns=2)
)


In [ ]:
by_mode = ntd_utils.calculate_efficiency_metrics_by_group(
    df, ["mode", "rtpa_name", "year"]
).pipe(
    make_long, 
    ["mode", "rtpa_name", "year"], 
    val_cols
)

#by_mode

In [ ]:
by_tos = ntd_utils.calculate_efficiency_metrics_by_group(
    df, ["type_of_service", "rtpa_name", "year"]
).pipe(
    make_long, 
    ["type_of_service", "rtpa_name", "year"], 
    val_cols
)

#by_tos

## Chart Ideas

* existing charts put cost-efficiency, service-effectiveness metrics in separate headings, and under those headings, there are by agency, by mode, by TOS
* re-organize this to put cost-efficiency, service-effectiveness metrics side-by-side for agency, then side-by-side for mode, side-by-side for TOS?

In [ ]:
test_tos= by_tos[by_tos.rtpa_name.str.contains("Del Norte")]#.variable.value_counts()

test_tos2 = test_tos.sort_values(["rtpa_name", "type_of_service", "variable", "year"]).groupby(
    ["rtpa_name", "type_of_service", "variable"]
).agg({
    "value": lambda x: list(x)
}).reset_index()

In [ ]:
from great_tables import GT
import gt_extras as gte
import polars as pl

In [ ]:
test_tos3 = pl.from_pandas(test_tos2)
(GT(test_tos3, rowname_col="variable")
    .fmt_nanoplot(columns="value")
)